# UAV C2 Dataset — Baseline ML Evaluation (Kaggle)

Notebook này triển khai kế hoạch `TEST PLAN — BASELINE ML EVALUATION FOR THE UAV C2 ATTACK DATASET`, phiên bản 1.0 (2026-10-12).

**Phạm vi kết quả chính:** phân loại nhị phân cửa sổ benign (`T0_Benign`) và DoS (`T4_DoS`), đặc trưng lấy từ PCAP C2, cửa sổ 1 giây, đánh giá ngoài theo **Leave-One-Mission-Out (LOMO)**. Các cửa sổ 2/5 giây và Leave-One-Channel-Scenario-Out là thí nghiệm phụ. Notebook không gộp các dataset radio tham khảo vào tập huấn luyện.

**Quan trọng:** thư mục dữ liệu chiến dịch chưa được đính kèm trong cuộc trò chuyện này; notebook chỉ tạo kết quả thật sau khi bạn thêm dữ liệu `data/campaign_results/` làm Kaggle Dataset và chạy notebook. Không có số liệu mô hình nào được giả lập hoặc điền sẵn.

Đầu ra được lưu trong `/kaggle/working/results/` và nén thành `uav_c2_baseline_results.zip` ở cuối notebook.

## Cấu trúc dữ liệu đầu vào cần có

```text
data/
└── campaign_results/
    ├── <run_001>/
    │   ├── metadata.json
    │   ├── c2_traffic.pcap        # hoặc đúng một file .pcap/.pcapng
    │   ├── attack_events.jsonl
    │   ├── flight_telemetry.csv   # nếu có
    │   └── twin_scope.json        # nếu có
    └── ...
```

Giữ nguyên thư mục run và các file JSON/PCAP. Nếu trường metadata khác với tên trường trong kế hoạch, hãy chạy tới cell kiểm kê, xem bảng `inventory`, rồi sửa phần alias ở cell kiểm kê; không tự đoán hoặc gán nhãn bằng tên run nếu dữ liệu metadata/event mâu thuẫn.

## 0. Cài thư viện

Trên Kaggle, bật Internet trong Notebook Settings nếu `scapy` chưa được cài sẵn. Các thư viện khoa học dữ liệu còn lại thường đã có sẵn trong môi trường Kaggle.

In [ ]:
%pip -q install scapy

## 1. Cấu hình thực nghiệm

Mặc định chạy baseline với tham số được khai báo trước (`TUNING_MODE="fixed"`) để tránh kéo dài thời gian chạy trên Kaggle. Có sẵn chế độ `grouped_cv` để bật tìm kiếm siêu tham số theo GroupKFold chỉ bên trong tập train của mỗi outer fold. Không đổi cấu hình sau khi xem kết quả outer test mà không ghi rõ vào change log.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, re, json, math, hashlib, platform, time, warnings, sys, subprocess, textwrap
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

SEED = 2026
EXPECTED_RUNS = 120
WINDOW_SIZES = [1, 2, 5]        # giây; 1s là thí nghiệm chính
PRIMARY_WINDOW = 1
GCS_IP = None                    # điền IP GCS CHỈ khi xác minh chắc chắn từ metadata/topology; None = bỏ feature theo hướng
HASH_RAW_INPUTS = True           # SHA-256 cho metadata, event, pcap, telemetry, twin scope
SAVE_MODEL_ARTIFACTS = True      # lưu fitted preprocessing + estimator cho từng outer fold
ALLOW_UNKNOWN_STATUS_AFTER_MANUAL_CHECK = False  # chỉ bật sau khi đã xác minh tất cả run hoàn tất từ log nguồn
TUNING_MODE = "fixed"            # "fixed" hoặc "grouped_cv"
RUN_SECONDARY_EVALUATION = True  # LOMO với cửa sổ 2/5s + leave-one-channel-out cho top K model
TOP_K_SECONDARY = 3
RUN_TELEMETRY_FUSION = False     # thí nghiệm tùy chọn; xem cell fusion và kiểm tra timestamp trước
RUN_ISOLATION_FOREST = False     # thí nghiệm novelty/anomaly riêng, không phải chứng cứ zero-day

INPUT_ROOT = Path("/kaggle/input")
CAMPAIGN_DIR_OVERRIDE = None     # ví dụ: "/kaggle/input/my-uav-c2-data/data/campaign_results"
OUTPUT_DIR = Path("/kaggle/working/results")
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if TUNING_MODE not in {"fixed", "grouped_cv"}:
    raise ValueError("TUNING_MODE phải là 'fixed' hoặc 'grouped_cv'.")
print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| NumPy:", np.__version__, "| scikit-learn:", sklearn.__version__)
print("Output directory:", OUTPUT_DIR)

## 2. Tìm thư mục campaign và kiểm kê run

Cell này tìm thư mục tên `campaign_results` trong Kaggle Input. Nếu có nhiều hơn một thư mục phù hợp, hãy đặt `CAMPAIGN_DIR_OVERRIDE` ở cell cấu hình để chọn đúng bộ dữ liệu. Inventory chưa chứng minh PCAP hợp lệ; nó chỉ kiểm tra cấu trúc và trường metadata cơ bản trước bước đọc PCAP.

In [ ]:
EXPECTED_MISSIONS = {"F1", "F4", "F7"}
EXPECTED_CHANNELS = {"C1_LOS", "C2_Moderate", "C3_NLOS", "C4_Dynamic"}
EXPECTED_ATTACKS = {"T0_Benign", "T4_DoS"}
BAD_STATUS_WORDS = {"failed", "failure", "error", "incomplete", "cancelled", "canceled", "aborted"}
GOOD_STATUS_WORDS = {"complete", "completed", "success", "succeeded", "finished", "done", "passed", "valid", "ok"}


def resolve_campaign_dir():
    if CAMPAIGN_DIR_OVERRIDE:
        p = Path(CAMPAIGN_DIR_OVERRIDE)
        if p.name == "campaign_results" and p.is_dir():
            return p
        candidates = [p / "data" / "campaign_results", p / "campaign_results"]
        for c in candidates:
            if c.is_dir():
                return c
        raise FileNotFoundError(f"Không tìm thấy campaign_results theo CAMPAIGN_DIR_OVERRIDE={p}")
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f"Không tìm thấy {INPUT_ROOT}. Hãy Add Input Kaggle Dataset trước.")
    found = []
    for p in INPUT_ROOT.rglob("campaign_results"):
        if p.is_dir() and any(p.rglob("metadata.json")):
            found.append(p)
    # unique paths
    found = sorted(set(found), key=lambda x: str(x))
    if len(found) == 0:
        raise FileNotFoundError(
            "Không tìm thấy thư mục campaign_results trong /kaggle/input. "
            "Hãy Add Input bộ dữ liệu hoặc sửa CAMPAIGN_DIR_OVERRIDE."
        )
    if len(found) > 1:
        raise RuntimeError("Có nhiều campaign_results. Hãy đặt CAMPAIGN_DIR_OVERRIDE rõ ràng:\n" + "\n".join(map(str, found)))
    return found[0]

CAMPAIGN_DIR = resolve_campaign_dir()
print("Campaign dir:", CAMPAIGN_DIR)


def norm_key(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())


def flatten_json(obj, prefix=""):
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            full = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, dict):
                out.extend(flatten_json(v, full))
            elif isinstance(v, list):
                # Preserve scalar lists but do not recursively treat array positions as metadata keys.
                out.append((full, v))
            else:
                out.append((full, v))
    return out


def meta_value(flat, aliases):
    aliases = {norm_key(a) for a in aliases}
    for full_key, value in flat:
        leaf = norm_key(full_key.split(".")[-1])
        if leaf in aliases and value is not None and str(value).strip() != "":
            return value
    return None


def first_regex(pattern, text):
    m = re.search(pattern, str(text), flags=re.I)
    return m.group(1) if m else None


def canonical_mission(raw, path_text):
    for v in (raw, path_text):
        got = first_regex(r"(?<![A-Za-z0-9])(F\d+)(?![A-Za-z0-9])", v)
        if got:
            return got.upper()
    return str(raw).strip() if raw is not None else None


def canonical_channel(raw, path_text):
    pat = r"(?<![A-Za-z0-9])(C\d+)[_-](LOS|Moderate|NLOS|Dynamic)(?![A-Za-z0-9])"
    for v in (raw, path_text):
        m = re.search(pat, str(v), flags=re.I)
        if m:
            suffix = {"los":"LOS", "moderate":"Moderate", "nlos":"NLOS", "dynamic":"Dynamic"}[m.group(2).lower()]
            return f"{m.group(1).upper()}_{suffix}"
    return str(raw).strip() if raw is not None else None


def _parse_attack_token(value):
    if value is None:
        return None
    text = str(value).strip()
    m = re.search(r"(?<![A-Za-z0-9])(T\d+)[_-]([A-Za-z0-9]+)(?![A-Za-z0-9])", text, flags=re.I)
    if m:
        stem = m.group(1).upper(); tail = m.group(2).lower()
        if stem == "T0" and tail == "benign": return "T0_Benign"
        if stem == "T4" and tail == "dos": return "T4_DoS"
        return f"{stem}_{m.group(2)}"
    low = text.lower()
    # Explicit metadata class takes precedence over parent folder names.
    if "benign" in low: return "T0_Benign"
    if re.search(r"\bdos\b", low) or "flood" in low: return "T4_DoS"
    return None


def canonical_attack(raw, path_text):
    parsed = _parse_attack_token(raw)
    if parsed is not None:
        return parsed
    parsed = _parse_attack_token(path_text)
    if parsed is not None:
        return parsed
    return str(raw).strip() if raw is not None else None


def canonical_seed(raw, path_text):
    if raw is not None:
        if isinstance(raw, (int, float, np.integer, np.floating)) and np.isfinite(raw):
            return str(int(raw)) if float(raw).is_integer() else str(raw)
        raw_text = str(raw).strip()
        if re.fullmatch(r"\d+", raw_text):
            return str(int(raw_text))
        m = re.search(r"(?:random[_ -]?seed|run[_ -]?seed|seed)[_ :=-]*(\d+)", raw_text, flags=re.I)
        if m: return m.group(1)
    m = re.search(r"(?:random[_ -]?seed|run[_ -]?seed|seed)[_ :=-]*(\d+)", str(path_text), flags=re.I)
    return m.group(1) if m else (str(raw).strip() if raw is not None else None)


def find_artifact(run_dir, exact_name, extensions=None):
    exact = list(run_dir.rglob(exact_name))
    if exact:
        return exact[0]
    if extensions:
        found = sorted([p for p in run_dir.rglob("*") if p.is_file() and p.suffix.lower() in extensions])
        if len(found) == 1:
            return found[0]
    return None

metadata_files = sorted(CAMPAIGN_DIR.rglob("metadata.json"))
# Only run directories that contain a plausible PCAP are considered campaign runs.
rows = []
for mp in metadata_files:
    run_dir = mp.parent
    pcap_path = find_artifact(run_dir, "c2_traffic.pcap", {".pcap", ".pcapng"})
    if pcap_path is None:
        # Keep an inventory row even when PCAP is missing; it will be excluded with a reason.
        pcap_candidates = [p for p in run_dir.rglob("*") if p.is_file() and p.suffix.lower() in {".pcap", ".pcapng"}]
        pcap_path = pcap_candidates[0] if len(pcap_candidates) == 1 else None
    try:
        meta = json.loads(mp.read_text(encoding="utf-8"))
        flat = flatten_json(meta)
        meta_error = ""
    except Exception as e:
        meta, flat, meta_error = {}, [], repr(e)
    path_text = str(run_dir.relative_to(CAMPAIGN_DIR))
    raw_mission = meta_value(flat, ["mission_id", "mission", "flight_mission", "route_id", "mission_name"])
    raw_channel = meta_value(flat, ["channel_id", "channel_scenario", "channel", "link_scenario", "channel_name"])
    raw_attack = meta_value(flat, ["attack_id", "traffic_label", "class_label", "attack_label", "attack_type", "condition_label"])
    raw_seed = meta_value(flat, ["seed", "random_seed", "run_seed"])
    raw_status = meta_value(flat, ["status", "run_status", "completion_status", "result_status"])
    ev = find_artifact(run_dir, "attack_events.jsonl")
    tel = find_artifact(run_dir, "flight_telemetry.csv")
    twin = find_artifact(run_dir, "twin_scope.json")
    status = str(raw_status).strip() if raw_status is not None else "UNKNOWN"
    status_bad = any(w in status.lower() for w in BAD_STATUS_WORDS)
    pcap_exists = pcap_path is not None and pcap_path.is_file() and pcap_path.stat().st_size > 0
    mission = canonical_mission(raw_mission, path_text)
    channel = canonical_channel(raw_channel, path_text)
    attack = canonical_attack(raw_attack, path_text)
    seed = canonical_seed(raw_seed, path_text)
    reasons = []
    if meta_error: reasons.append("metadata_json_invalid")
    if not pcap_exists: reasons.append("missing_or_ambiguous_or_empty_pcap")
    if mission is None: reasons.append("missing_mission_id")
    if channel is None: reasons.append("missing_channel_id")
    if attack not in EXPECTED_ATTACKS: reasons.append("attack_id_not_in_T0_Benign_T4_DoS")
    if seed is None: reasons.append("missing_seed")
    if status_bad:
        reasons.append(f"run_status={status}")
    elif status.upper() == "UNKNOWN" and not ALLOW_UNKNOWN_STATUS_AFTER_MANUAL_CHECK:
        reasons.append("missing_run_status_manual_review_required")
    elif status.upper() != "UNKNOWN" and not any(w in status.lower() for w in GOOD_STATUS_WORDS):
        reasons.append(f"unrecognized_run_status_manual_review_required:{status}")
    rows.append({
        "run_id": path_text, "run_dir": str(run_dir), "metadata_path": str(mp),
        "pcap_path": str(pcap_path) if pcap_path else "", "attack_events_path": str(ev) if ev else "",
        "telemetry_path": str(tel) if tel else "", "twin_scope_path": str(twin) if twin else "",
        "mission_id": mission, "channel_id": channel, "attack_id": attack, "seed": seed,
        "metadata_status": status, "metadata_parse_error": meta_error,
        "pcap_bytes": int(pcap_path.stat().st_size) if pcap_exists else 0,
        "has_events": bool(ev and ev.is_file() and ev.stat().st_size > 0),
        "has_telemetry": bool(tel and tel.is_file() and tel.stat().st_size > 0),
        "has_twin_scope": bool(twin and twin.is_file() and twin.stat().st_size > 0),
        "inventory_reasons": ";".join(reasons),
        "inventory_candidate": len(reasons) == 0,
    })

inventory = pd.DataFrame(rows)
if inventory.empty:
    raise RuntimeError(f"Tìm thấy {len(metadata_files)} metadata.json nhưng không tạo được inventory run.")
inventory = inventory.sort_values("run_id").reset_index(drop=True)
inventory.to_csv(OUTPUT_DIR / "dataset_inventory.csv", index=False)
print("Metadata files:", len(metadata_files), "| Inventory rows:", len(inventory))
display(inventory[["run_id", "mission_id", "channel_id", "attack_id", "seed", "metadata_status", "pcap_bytes", "inventory_reasons"]].head(15))
print("\nSố run theo lớp:")
display(inventory.groupby("attack_id", dropna=False).size().rename("n_runs").reset_index())
print("\nSố run theo mission × channel × lớp:")
run_counts = inventory.groupby(["mission_id", "channel_id", "attack_id"], dropna=False).size().rename("n_runs").reset_index()
run_counts.to_csv(OUTPUT_DIR / "dataset_design_run_counts.csv", index=False)
display(run_counts)

### 2.1. Quality gate cho thiết kế dự kiến

Thiết kế danh nghĩa của kế hoạch là `3 mission × 4 channel × 2 class × 5 seed = 120 runs`. Cell này **báo cáo chênh lệch**, không tự tạo run bị thiếu và không tự thay đổi nhãn. Kiểm tra một vài dòng metadata và tên run trước khi tiếp tục nếu trường nào bị `UNKNOWN` hoặc không khớp.

In [ ]:
print(f"Expected nominal runs: {EXPECTED_RUNS}; observed metadata records: {len(inventory)}")
if len(inventory) != EXPECTED_RUNS:
    print("WARNING: số metadata record khác 120. Hãy giải thích run thiếu/trùng trong phần Data Quality của bài báo.")
print("Mission values:", sorted(inventory["mission_id"].dropna().astype(str).unique().tolist()))
print("Channel values:", sorted(inventory["channel_id"].dropna().astype(str).unique().tolist()))
print("Attack values:", sorted(inventory["attack_id"].dropna().astype(str).unique().tolist()))
print("Seed values:", sorted(inventory["seed"].dropna().astype(str).unique().tolist()))
print("Metadata status values:", inventory["metadata_status"].value_counts(dropna=False).to_dict())
if (inventory["metadata_status"].astype(str).str.upper() == "UNKNOWN").any():
    print("WARNING: một số run thiếu trường status rõ ràng; đối chiếu metadata/log nguồn trước khi chốt accepted runs.")

combo_cols = ["mission_id", "channel_id", "attack_id", "seed"]
if inventory[combo_cols].notna().all(axis=1).any():
    dup = inventory[inventory[combo_cols].notna().all(axis=1)].duplicated(combo_cols, keep=False)
    if dup.any():
        print("WARNING: có tổ hợp mission/channel/attack/seed trùng:")
        display(inventory.loc[dup, ["run_id"] + combo_cols])

factor_warnings = []
if set(inventory["mission_id"].dropna()) != EXPECTED_MISSIONS:
    factor_warnings.append("mission IDs không khớp F1/F4/F7")
if set(inventory["channel_id"].dropna()) != EXPECTED_CHANNELS:
    factor_warnings.append("channel IDs không khớp 4 kịch bản dự kiến")
if set(inventory["attack_id"].dropna()) != EXPECTED_ATTACKS:
    factor_warnings.append("attack IDs không khớp đúng T0_Benign/T4_DoS")
if factor_warnings:
    print("\nCẦN KIỂM TRA TRƯỚC KHI TRAIN:")
    for w in factor_warnings: print(" -", w)
else:
    print("Các mức factor cơ bản khớp thiết kế trong test plan.")

## 3. Đọc attack event và trích xuất PCAP thành cửa sổ

Các quyết định được khóa trong extractor:

- Bin căn theo epoch: `[t, t + W)`; chỉ giữ cửa sổ đủ `W` giây nằm trong khoảng timestamp đầu/cuối của PCAP để tránh tính tốc độ trên cửa sổ biên bị thiếu thời lượng. Vì timestamp đầu/cuối gói không phải lúc nào cũng là thời điểm bắt đầu/kết thúc capture thực tế, đây là kiểm tra bảo thủ cần ghi rõ trong báo cáo.
- Nhãn DoS = 1 khi hợp của các khoảng attack event chồng lấn **ít nhất 50%** độ dài cửa sổ; benign = 0. Không gán toàn bộ run DoS thành lớp 1.
- Inter-arrival features bằng 0 khi có dưới 2 packet; size features bằng 0 khi cửa sổ rỗng. `packet_count` và `total_bytes` cho phép mô hình phân biệt các cửa sổ rỗng.
- IP/port thô, payload, epoch tuyệt đối, run/mission/channel/seed, attack-event metadata không nằm trong feature matrix.
- Các directional features chỉ thêm khi bạn khai báo `GCS_IP` đã được xác minh. Nếu chưa chắc IP GCS, để `None`.

In [ ]:
from scapy.utils import PcapReader, PcapNgReader
from scapy.layers.inet import IP, TCP, UDP, ICMP
from scapy.layers.inet6 import IPv6, ICMPv6

KNOWN_IP_PROTOS = [1, 2, 6, 17, 41, 47, 50, 58]
TIME_ROUND_DIGITS = 6


def recursively_find_value(obj, aliases):
    aliases = {norm_key(a) for a in aliases}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if norm_key(k) in aliases and v is not None and str(v).strip() != "":
                return v
        for v in obj.values():
            got = recursively_find_value(v, aliases)
            if got is not None:
                return got
    elif isinstance(obj, list):
        for v in obj:
            got = recursively_find_value(v, aliases)
            if got is not None:
                return got
    return None


def read_attack_intervals(path):
    """Return event intervals [start,end) in epoch seconds plus parsing diagnostics."""
    intervals, errors, records = [], [], 0
    if not path or not Path(path).is_file():
        return intervals, ["missing_attack_events_file"], records
    p = Path(path)
    with p.open("r", encoding="utf-8", errors="replace") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            records += 1
            try:
                rec = json.loads(line)
                start = recursively_find_value(rec, ["start_epoch_s", "start_epoch", "attack_start_epoch_s"])
                end = recursively_find_value(rec, ["end_epoch_s", "end_epoch", "attack_end_epoch_s"])
                if start is None or end is None:
                    continue
                start, end = float(start), float(end)
                if np.isfinite(start) and np.isfinite(end) and end > start:
                    intervals.append((start, end))
                else:
                    errors.append(f"invalid_interval_line_{line_no}")
            except Exception as e:
                errors.append(f"jsonl_line_{line_no}:{type(e).__name__}")
    intervals.sort()
    merged = []
    for s, e in intervals:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)
    return [(float(s), float(e)) for s, e in merged], errors, records


def endpoint_repr(ip_addr, port, transport):
    if ip_addr is None:
        return None
    if transport in {"tcp", "udp"}:
        return (str(ip_addr), int(port) if port is not None else -1, transport)
    return (str(ip_addr), -1, transport)


def read_pcap_records(pcap_path, gcs_ip=None):
    """Read one PCAP/PCAPNG and return packet-level primitives (no payload or raw IDs in X)."""
    p = Path(pcap_path)
    reader = PcapNgReader(str(p)) if p.suffix.lower() == ".pcapng" else PcapReader(str(p))
    linktype = getattr(reader, "linktype", None)
    records, parse_errors, truncated_count = [], 0, 0
    try:
        while True:
            try:
                pkt = next(reader)
            except StopIteration:
                break
            except Exception:
                parse_errors += 1
                # A corrupt record may prevent the reader from resynchronizing; stop rather than silently skip arbitrary bytes.
                break
            try:
                ts = round(float(pkt.time), TIME_ROUND_DIGITS)
                cap_len = int(len(pkt))
                wire_len = int(getattr(pkt, "wirelen", cap_len) or cap_len)
                if wire_len > cap_len:
                    truncated_count += 1

                src_ip = dst_ip = None
                ip_proto = None
                if IP in pkt:
                    lyr = pkt[IP]
                    src_ip, dst_ip, ip_proto = str(lyr.src), str(lyr.dst), int(lyr.proto)
                elif IPv6 in pkt:
                    lyr = pkt[IPv6]
                    src_ip, dst_ip, ip_proto = str(lyr.src), str(lyr.dst), int(lyr.nh)

                sport = dport = None
                if TCP in pkt:
                    trans, sport, dport = "tcp", int(pkt[TCP].sport), int(pkt[TCP].dport)
                elif UDP in pkt:
                    trans, sport, dport = "udp", int(pkt[UDP].sport), int(pkt[UDP].dport)
                elif ICMP in pkt or ICMPv6 in pkt:
                    trans = "icmp"
                else:
                    trans = "other"

                src_ep = endpoint_repr(src_ip, sport, trans)
                dst_ep = endpoint_repr(dst_ip, dport, trans)
                flow5 = (src_ip, sport if sport is not None else 0, dst_ip, dport if dport is not None else 0, ip_proto) if src_ip and dst_ip else None
                direction = None
                if gcs_ip:
                    if src_ip == str(gcs_ip): direction = "outbound"
                    elif dst_ip == str(gcs_ip): direction = "inbound"
                    else: direction = "other"

                records.append({
                    "timestamp": ts, "captured_len": cap_len, "wire_len": wire_len,
                    "ip_proto": ip_proto, "transport": trans, "src_endpoint": src_ep,
                    "dst_endpoint": dst_ep, "flow5": flow5, "direction": direction,
                })
            except Exception:
                parse_errors += 1
    finally:
        try: reader.close()
        except Exception: pass

    if not records:
        raise ValueError(f"Không đọc được packet hợp lệ từ {p} (parse_errors={parse_errors}).")
    raw_times = np.asarray([r["timestamp"] for r in records], dtype=float)
    backward_jumps = int(np.sum(np.diff(raw_times) < 0)) if len(raw_times) > 1 else 0
    packet_df = pd.DataFrame(records).sort_values("timestamp").reset_index(drop=True)
    ordered_diffs = np.diff(packet_df["timestamp"].to_numpy(dtype=float)) if len(packet_df) > 1 else np.array([])
    proto_counts = packet_df["ip_proto"].fillna(-1).astype(int).value_counts().sort_index().to_dict()
    transport_counts = packet_df["transport"].value_counts().to_dict()
    protocol_coverage = json.dumps({"ip_proto_numeric_-1_is_non_ip": proto_counts, "transport": transport_counts}, sort_keys=True)
    quality = {
        "pcap_linktype": linktype,
        "pcap_packet_count": int(len(packet_df)),
        "pcap_first_epoch_s": float(packet_df["timestamp"].min()),
        "pcap_last_epoch_s": float(packet_df["timestamp"].max()),
        "pcap_duration_s": float(packet_df["timestamp"].max() - packet_df["timestamp"].min()),
        "pcap_parse_errors": int(parse_errors),
        "pcap_truncated_packet_count": int(truncated_count),
        "pcap_non_ip_packet_count": int(packet_df["ip_proto"].isna().sum()),
        "pcap_exact_duplicate_record_count": int(packet_df.duplicated(keep="first").sum()),
        "pcap_timestamp_backward_jump_count": backward_jumps,
        "pcap_max_inter_packet_gap_s": float(np.max(ordered_diffs)) if len(ordered_diffs) else 0.0,
        "pcap_p99_inter_packet_gap_s": float(np.quantile(ordered_diffs, 0.99)) if len(ordered_diffs) else 0.0,
        "pcap_protocol_coverage_json": protocol_coverage,
        "pcap_bytes_on_disk": int(p.stat().st_size),
    }
    return packet_df, quality


def interval_overlap_fraction(start, window_s, intervals):
    end = start + window_s
    overlap = 0.0
    for ev_start, ev_end in intervals:
        overlap += max(0.0, min(end, ev_end) - max(start, ev_start))
    # Intervals are merged before use, so overlap cannot be double-counted.
    return min(1.0, overlap / window_s)


def summarize_numeric(arr, prefix):
    arr = np.asarray(arr, dtype=float)
    if len(arr) == 0:
        return {f"{prefix}_{s}": 0.0 for s in ["mean", "std", "min", "max", "median", "q25", "q75", "q90"]}
    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr, ddof=0)),
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_max": float(np.max(arr)),
        f"{prefix}_median": float(np.median(arr)),
        f"{prefix}_q25": float(np.quantile(arr, 0.25)),
        f"{prefix}_q75": float(np.quantile(arr, 0.75)),
        f"{prefix}_q90": float(np.quantile(arr, 0.90)),
    }


def feature_row_for_window(packet_subset, start, window_s, intervals, attack_id):
    n = len(packet_subset)
    row = {
        "window_start_epoch_s": float(start),
        "window_size_s": int(window_s),
        "packet_count": int(n),
        "total_bytes": int(packet_subset["captured_len"].sum()) if n else 0,
        "packets_per_second": float(n / window_s),
        "bytes_per_second": float(packet_subset["captured_len"].sum() / window_s) if n else 0.0,
    }
    row.update(summarize_numeric(packet_subset["captured_len"].to_numpy() if n else [], "captured_len"))
    row.update(summarize_numeric(packet_subset["wire_len"].to_numpy() if n else [], "wire_len"))
    ts = np.sort(packet_subset["timestamp"].to_numpy(dtype=float)) if n else np.array([])
    iat = np.diff(ts) if len(ts) >= 2 else np.array([])
    row.update(summarize_numeric(iat, "iat"))

    ipvals = packet_subset["ip_proto"] if n else pd.Series(dtype=float)
    for proto in KNOWN_IP_PROTOS:
        row[f"ipproto_{proto}_count"] = int((ipvals == proto).sum())
    known_mask = ipvals.isin(KNOWN_IP_PROTOS) if n else pd.Series(dtype=bool)
    row["ipproto_other_count"] = int((~known_mask).sum()) if n else 0
    for name in ["udp", "tcp", "icmp", "other"]:
        row[f"transport_{name}_count"] = int((packet_subset["transport"] == name).sum()) if n else 0
    denom = max(n, 1)
    for name in ["udp", "tcp", "icmp", "other"]:
        row[f"transport_{name}_prop"] = row[f"transport_{name}_count"] / denom
    for proto in KNOWN_IP_PROTOS:
        row[f"ipproto_{proto}_prop"] = row[f"ipproto_{proto}_count"] / denom
    row["ipproto_other_prop"] = row["ipproto_other_count"] / denom

    row["unique_src_endpoints"] = int(packet_subset["src_endpoint"].dropna().nunique()) if n else 0
    row["unique_dst_endpoints"] = int(packet_subset["dst_endpoint"].dropna().nunique()) if n else 0
    row["unique_5tuples"] = int(packet_subset["flow5"].dropna().nunique()) if n else 0
    if GCS_IP:
        for direction in ["inbound", "outbound", "other"]:
            mask = packet_subset["direction"] == direction if n else pd.Series(dtype=bool)
            count = int(mask.sum()) if n else 0
            nbytes = int(packet_subset.loc[mask, "captured_len"].sum()) if n else 0
            row[f"direction_{direction}_packet_count"] = count
            row[f"direction_{direction}_packet_prop"] = count / denom
            row[f"direction_{direction}_bytes"] = nbytes
            row[f"direction_{direction}_bytes_prop"] = nbytes / max(row["total_bytes"], 1)

    frac = interval_overlap_fraction(start, window_s, intervals) if attack_id == "T4_DoS" else 0.0
    row["attack_overlap_fraction"] = frac  # retained temporarily for QC, excluded from X
    row["y"] = int(frac >= 0.5) if attack_id == "T4_DoS" else 0
    return row


def extract_features_for_run(inv_row, window_sizes=WINDOW_SIZES):
    run_id = inv_row["run_id"]
    attack_id = inv_row["attack_id"]
    event_path = inv_row["attack_events_path"]
    intervals, event_errors, event_line_count = read_attack_intervals(event_path if event_path else None)
    packet_df, quality = read_pcap_records(inv_row["pcap_path"], GCS_IP)
    if attack_id == "T4_DoS":
        if not intervals:
            raise ValueError("T4_DoS run không có attack interval hợp lệ trong attack_events.jsonl.")
        first_ts, last_ts = quality["pcap_first_epoch_s"], quality["pcap_last_epoch_s"]
        if not any(e > first_ts and s < last_ts for s, e in intervals):
            raise ValueError("attack event không chồng lấn timestamp PCAP; cần kiểm tra clock/epoch trước khi gán nhãn.")
    else:
        intervals = []  # benign windows always y=0; event file is not used as a label source

    quality["event_record_count"] = int(event_line_count)
    quality["event_interval_count"] = int(len(intervals))
    quality["event_parse_warnings"] = ";".join(event_errors)
    quality["run_id"] = run_id

    all_rows = []
    min_ts, max_ts = quality["pcap_first_epoch_s"], quality["pcap_last_epoch_s"]
    for W in window_sizes:
        # Only complete epoch-aligned windows are retained. Indices represent [idx*W, (idx+1)*W).
        first_idx = int(math.ceil(min_ts / W - 1e-10))
        last_idx = int(math.floor((max_ts - W) / W + 1e-10))
        if last_idx < first_idx:
            quality[f"complete_windows_{W}s"] = 0
            quality[f"dropped_edge_or_short_windows_{W}s"] = 1
            continue
        p = packet_df.copy()
        p["bin_idx"] = np.floor(np.round(p["timestamp"].to_numpy(dtype=float), TIME_ROUND_DIGITS) / W).astype(int)
        groups = {int(k): g for k, g in p.groupby("bin_idx", sort=False)}
        windows_created = 0
        for idx in range(first_idx, last_idx + 1):
            start = float(idx * W)
            subset = groups.get(idx, packet_df.iloc[0:0])
            row = feature_row_for_window(subset, start, W, intervals, attack_id)
            row.update({
                "run_id": run_id,
                "mission_id": inv_row["mission_id"],
                "channel_id": inv_row["channel_id"],
                "attack_id": attack_id,
                "seed": str(inv_row["seed"]),
                "condition_group": f"{inv_row['mission_id']}|{inv_row['channel_id']}|{inv_row['seed']}",
            })
            all_rows.append(row)
            windows_created += 1
        quality[f"complete_windows_{W}s"] = int(windows_created)
        quality[f"dropped_edge_or_short_windows_{W}s"] = int((last_idx - first_idx + 1) == 0)
    return all_rows, quality

print("Extractor definitions loaded. No model has been trained yet.")

## 4. Audit PCAP, tạo features 1/2/5 giây, ghi lý do loại run

Quá trình đọc PCAP có thể mất thời gian. Nó chạy một lần cho mỗi run, rồi tạo cả ba độ dài cửa sổ từ cùng luồng packet đã đọc. Những run thiếu event hợp lệ hoặc lệch timestamp sẽ bị ghi vào quality report, không bị âm thầm gán nhãn.

In [ ]:
def sha256_file(path, chunk_bytes=4 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk: break
            h.update(chunk)
    return h.hexdigest()

# Hash the actual source artifacts for reproducibility. Large PCAPs can make this step I/O intensive.
if HASH_RAW_INPUTS:
    hash_rows = []
    for _, r in inventory.iterrows():
        for col in ["metadata_path", "pcap_path", "attack_events_path", "telemetry_path", "twin_scope_path"]:
            value = r.get(col, "")
            if value and Path(str(value)).is_file():
                try:
                    hash_rows.append({"run_id": r["run_id"], "artifact_type": col, "path": str(value),
                                     "bytes": Path(value).stat().st_size, "sha256": sha256_file(value)})
                except Exception as e:
                    hash_rows.append({"run_id": r["run_id"], "artifact_type": col, "path": str(value),
                                     "bytes": Path(value).stat().st_size, "sha256": "", "hash_error": repr(e)})
    hashes_df = pd.DataFrame(hash_rows)
    hashes_df.to_csv(OUTPUT_DIR / "input_hashes.csv", index=False)
    print("Hashed source artifacts:", len(hashes_df))
else:
    hashes_df = pd.DataFrame()
    print("HASH_RAW_INPUTS=False: bỏ qua SHA-256 raw inputs; ghi lý do trong log khi báo cáo.")

feature_rows, quality_rows, excluded_rows = [], [], []
for i, r in inventory.iterrows():
    # Hard preflight gate: do not infer missing factors or process a run whose status is explicitly failed.
    if not bool(r["inventory_candidate"]):
        excluded_rows.append({"run_id": r["run_id"], "reason": r["inventory_reasons"] or "preflight_failed"})
        continue
    try:
        fr, qr = extract_features_for_run(r, WINDOW_SIZES)
        feature_rows.extend(fr)
        quality_rows.append(qr)
        if not fr:
            excluded_rows.append({"run_id": r["run_id"], "reason": "no_complete_windows_for_any_requested_window_size"})
        if (i + 1) % 10 == 0:
            print(f"Processed inventory row {i+1}/{len(inventory)} | windows extracted: {len(feature_rows):,}")
    except Exception as e:
        excluded_rows.append({"run_id": r["run_id"], "reason": f"extraction_error:{type(e).__name__}:{e}"})
        print(f"EXCLUDED {r['run_id']}: {type(e).__name__}: {e}")

features_all = pd.DataFrame(feature_rows)
quality_df = pd.DataFrame(quality_rows)
excluded_df = pd.DataFrame(excluded_rows, columns=["run_id", "reason"])
quality_df.to_csv(OUTPUT_DIR / "pcap_quality_by_run.csv", index=False)
excluded_df.to_csv(OUTPUT_DIR / "excluded_runs.csv", index=False)
if features_all.empty:
    raise RuntimeError("Không tạo được feature window nào. Kiểm tra inventory, PCAP timestamp, event interval và log lỗi phía trên.")

# Exact overlap fraction is exported separately as label-audit evidence, never alongside model features.
label_audit_cols = [c for c in ["run_id", "mission_id", "channel_id", "attack_id", "seed", "condition_group", "window_start_epoch_s", "window_size_s", "attack_overlap_fraction", "y"] if c in features_all.columns]
features_all[label_audit_cols].to_csv(OUTPUT_DIR / "window_label_audit.csv", index=False)

for W in WINDOW_SIZES:
    out = OUTPUT_DIR / f"features_{W}s.csv"
    part = features_all[features_all["window_size_s"] == W].copy()
    part = part.drop(columns=["attack_overlap_fraction"], errors="ignore")
    part.to_csv(out, index=False)
    print(f"{out.name}: {len(part):,} windows | runs={part['run_id'].nunique()} | labels={part['y'].value_counts(dropna=False).to_dict()}")

print("\nRun exclusions:", len(excluded_df))
display(excluded_df.head(30))

## 5. Báo cáo chất lượng dữ liệu và phân phối nhãn

Trước khi train, xem kỹ các bảng sau. Với từng `T4_DoS`, phải có event interval hợp lệ chồng lấn PCAP và có cả cửa sổ y=0 trước/sau event nếu capture đủ dài. Nếu tất cả cửa sổ của DoS run thành y=1, hãy kiểm tra timestamp/event interval trước; đây thường là dấu hiệu nhãn chưa phản ánh đúng thời gian attack.

In [ ]:
# Run-level counts, expected-design counts, and per-window class distribution.
quality_report_path = OUTPUT_DIR / "data_quality_report.txt"
lines = []
lines.append("UAV C2 DATA QUALITY REPORT")
lines.append(f"Generated UTC: {datetime.now(timezone.utc).isoformat()}")
lines.append(f"Campaign dir: {CAMPAIGN_DIR}")
lines.append(f"Metadata records: {len(inventory)} (nominal expected={EXPECTED_RUNS})")
lines.append(f"Runs with at least one extracted window: {features_all['run_id'].nunique()}")
lines.append(f"Excluded runs: {len(excluded_df)}")
lines.append(f"Metadata status counts: {inventory['metadata_status'].value_counts(dropna=False).to_dict()}")
lines.append(f"SHA256 enabled: {HASH_RAW_INPUTS}")
lines.append("\nInventory exclusion reasons:")
if len(excluded_df):
    lines.extend((excluded_df["reason"].value_counts(dropna=False).to_string()).splitlines())
else:
    lines.append("None recorded.")
lines.append("\nObserved factors:")
for col in ["mission_id", "channel_id", "attack_id", "seed"]:
    lines.append(f"{col}: {sorted(inventory[col].dropna().astype(str).unique().tolist())}")
lines.append("\nPer-window label counts:")
for W in WINDOW_SIZES:
    d = features_all[features_all["window_size_s"] == W]
    lines.append(f"{W}s: windows={len(d)}; runs={d['run_id'].nunique()}; labels={d['y'].value_counts(dropna=False).to_dict()}")
if not quality_df.empty:
    lines.append("\nPCAP quality summary:")
    for col in ["pcap_packet_count", "pcap_duration_s", "pcap_parse_errors", "pcap_truncated_packet_count", "pcap_non_ip_packet_count", "pcap_exact_duplicate_record_count", "pcap_timestamp_backward_jump_count", "pcap_max_inter_packet_gap_s", "pcap_p99_inter_packet_gap_s"]:
        if col in quality_df:
            lines.append(f"{col}: min={quality_df[col].min()}, median={quality_df[col].median()}, max={quality_df[col].max()}")
report_text = "\n".join(lines)
quality_report_path.write_text(report_text, encoding="utf-8")
print(report_text)

_all_window_label_distribution = []
_per_run_label_audits = []
for W in WINDOW_SIZES:
    d = features_all[features_all["window_size_s"] == W]
    if d.empty: continue
    print(f"\n{W}s — label distribution by mission × channel × attack_id")
    wdist = d.groupby(["window_size_s", "mission_id", "channel_id", "attack_id", "y"], dropna=False).size().rename("n_windows").reset_index()
    # Save once per iteration, including all previously calculated window sizes.
    if "_all_window_label_distribution" not in globals():
        _all_window_label_distribution = []
    _all_window_label_distribution.append(wdist)
    display(wdist)
    run_class_counts = d.groupby(["run_id", "attack_id", "y"]).size().unstack("y", fill_value=0)
    print(f"{W}s — count of runs with positive and negative windows (columns 0/1 may be absent):")
    display(run_class_counts.head(20))
    per_run_labels = d.groupby(["run_id", "attack_id"])["y"].agg(["count", "sum", "nunique"]).reset_index()
    per_run_labels["n_negative_windows"] = per_run_labels["count"] - per_run_labels["sum"]
    per_run_labels["label_warning"] = np.where(
        (per_run_labels["attack_id"] == "T4_DoS") & (per_run_labels["sum"] == 0), "T4_DoS_has_no_positive_window",
        np.where((per_run_labels["attack_id"] == "T4_DoS") & (per_run_labels["n_negative_windows"] == 0), "T4_DoS_has_no_negative_window", "")
    )
    per_run_labels["window_size_s"] = W
    if "_per_run_label_audits" not in globals(): _per_run_label_audits = []
    _per_run_label_audits.append(per_run_labels)
    if (per_run_labels["label_warning"] != "").any():
        print("WARNING label audit — kiểm tra thời gian capture/event:")
        display(per_run_labels[per_run_labels["label_warning"] != ""])

# Basic numeric feature sanity check; provenance/labels are not candidate model features.
PROVENANCE_COLS = {
    "window_start_epoch_s", "window_size_s", "run_id", "mission_id", "channel_id", "attack_id", "seed",
    "condition_group", "attack_overlap_fraction", "y", "label_name", "source_pcap", "status", "metadata_status"
}
primary_df = features_all[features_all["window_size_s"] == PRIMARY_WINDOW].copy()
feature_cols = [c for c in primary_df.columns if c not in PROVENANCE_COLS and pd.api.types.is_numeric_dtype(primary_df[c])]
if not feature_cols:
    raise RuntimeError("Không xác định được cột feature numeric.")
Xcheck = primary_df[feature_cols].replace([np.inf, -np.inf], np.nan)
qc_feature = pd.DataFrame({
    "feature": feature_cols,
    "missing_fraction": [float(Xcheck[c].isna().mean()) for c in feature_cols],
    "n_unique": [int(Xcheck[c].nunique(dropna=True)) for c in feature_cols],
    "is_constant_or_all_missing": [bool(Xcheck[c].nunique(dropna=True) <= 1) for c in feature_cols],
})
qc_feature.to_csv(OUTPUT_DIR / "feature_quality.csv", index=False)
if "_all_window_label_distribution" in globals() and _all_window_label_distribution:
    pd.concat(_all_window_label_distribution, ignore_index=True).to_csv(OUTPUT_DIR / "window_label_distribution.csv", index=False)
if "_per_run_label_audits" in globals() and _per_run_label_audits:
    pd.concat(_per_run_label_audits, ignore_index=True).to_csv(OUTPUT_DIR / "run_label_audit.csv", index=False)
accepted_ids = set(quality_df["run_id"].astype(str)) if not quality_df.empty else set()
accepted_inventory = inventory[inventory["run_id"].astype(str).isin(accepted_ids)].copy()
accepted_inventory.to_csv(OUTPUT_DIR / "accepted_runs.csv", index=False)
# Dataset design table at full mission × channel × class × seed granularity.
design_table = inventory.groupby(["mission_id", "channel_id", "attack_id", "seed"], dropna=False).agg(n_observed_runs=("run_id", "nunique")).reset_index()
accepted_counts = accepted_inventory.groupby(["mission_id", "channel_id", "attack_id", "seed"], dropna=False).agg(n_accepted_runs=("run_id", "nunique")).reset_index()
design_table = design_table.merge(accepted_counts, on=["mission_id", "channel_id", "attack_id", "seed"], how="left")
design_table["n_accepted_runs"] = design_table["n_accepted_runs"].fillna(0).astype(int)
design_table.to_csv(OUTPUT_DIR / "dataset_design_table.csv", index=False)
print(f"\nPrimary windows={len(primary_df):,}; feature count={len(feature_cols)}")
print("All-missing features:", qc_feature.loc[qc_feature["n_unique"] == 0, "feature"].tolist())
print("Constant/all-missing feature count (reported, not selected using test labels):", int(qc_feature["is_constant_or_all_missing"].sum()))

## 6. Định nghĩa baseline và metric

- Bộ chính: Dummy-most-frequent, Logistic Regression, Decision Tree, Random Forest, Extra Trees, RBF-SVM, kNN, Gaussian Naive Bayes, HistGradientBoosting.
- Dùng median imputation trong pipeline; LR/SVM/kNN có StandardScaler. Class weight chỉ dùng ở model hỗ trợ. Không oversample trước khi split.
- Chỉ `y` là nhãn. Toàn bộ `run_id`, `mission_id`, `channel_id`, `seed`, `condition_group`, epoch và overlap attack bị loại khỏi `X`.
- Metric chính: DoS precision/recall/F1, macro-F1, balanced accuracy, average precision (PR-AUC), FPR, confusion counts, batch prediction latency và throughput.

In [ ]:
from sklearn.base import clone
import joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import (
    precision_score, recall_score, f1_score, balanced_accuracy_score,
    average_precision_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, PrecisionRecallDisplay
)

MODEL_NAMES = [
    "Dummy-most-frequent", "LogisticRegression", "DecisionTree", "RandomForest",
    "ExtraTrees", "RBF-SVM", "kNN", "GaussianNB", "HistGradientBoosting"
]


def build_model(name, for_search=False, n_train=None):
    n_jobs = 1 if for_search else -1
    if name in {"LogisticRegression", "RBF-SVM", "kNN"}:
        steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    else:
        steps = [("imputer", SimpleImputer(strategy="median"))]
    if name == "Dummy-most-frequent":
        clf = DummyClassifier(strategy="most_frequent")
    elif name == "LogisticRegression":
        clf = LogisticRegression(C=1.0, penalty="l2", solver="liblinear", class_weight="balanced", max_iter=2000, random_state=SEED)
    elif name == "DecisionTree":
        clf = DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=SEED)
    elif name == "RandomForest":
        clf = RandomForestClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=2,
                                    class_weight="balanced_subsample", random_state=SEED, n_jobs=n_jobs)
    elif name == "ExtraTrees":
        clf = ExtraTreesClassifier(n_estimators=200, max_features="sqrt", min_samples_leaf=2,
                                  class_weight="balanced", random_state=SEED, n_jobs=n_jobs)
    elif name == "RBF-SVM":
        clf = SVC(C=1.0, kernel="rbf", gamma="scale", class_weight="balanced", probability=False, cache_size=1000, random_state=SEED)
    elif name == "kNN":
        k = min(5, max(1, int(n_train or 5)))
        clf = KNeighborsClassifier(n_neighbors=k, weights="distance", n_jobs=n_jobs)
    elif name == "GaussianNB":
        clf = GaussianNB()
    elif name == "HistGradientBoosting":
        clf = HistGradientBoostingClassifier(max_iter=150, max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)
    else:
        raise KeyError(name)
    steps.append(("clf", clf))
    return Pipeline(steps)


PARAM_GRIDS = {
    "LogisticRegression": {"clf__C": [0.1, 1.0, 10.0]},
    "DecisionTree": {"clf__max_depth": [8, 12, None], "clf__min_samples_leaf": [2, 5]},
    "RandomForest": {"clf__min_samples_leaf": [1, 2, 4], "clf__max_features": ["sqrt", 0.7]},
    "ExtraTrees": {"clf__min_samples_leaf": [1, 2, 4], "clf__max_features": ["sqrt", 0.7]},
    "RBF-SVM": {"clf__C": [0.5, 1.0, 5.0], "clf__gamma": ["scale"]},
    "kNN": {"clf__n_neighbors": [3, 5, 9], "clf__weights": ["uniform", "distance"]},
    "HistGradientBoosting": {"clf__max_leaf_nodes": [15, 31], "clf__l2_regularization": [0.0, 1.0]},
}


def continuous_scores(estimator, X):
    clf = estimator.named_steps["clf"] if hasattr(estimator, "named_steps") else estimator
    if hasattr(estimator, "predict_proba"):
        p = estimator.predict_proba(X)
        classes = list(getattr(clf, "classes_", []))
        if 1 in classes:
            return np.asarray(p)[:, classes.index(1)]
        return np.zeros(len(X), dtype=float)
    if hasattr(estimator, "decision_function"):
        z = np.asarray(estimator.decision_function(X))
        if z.ndim == 1:
            classes = list(getattr(clf, "classes_", []))
            return z if classes == [0, 1] else z
        classes = list(getattr(clf, "classes_", []))
        return z[:, classes.index(1)] if 1 in classes else np.zeros(len(X), dtype=float)
    return np.asarray(estimator.predict(X), dtype=float)


def compute_metrics(y_true, y_pred, y_score, predict_seconds):
    y_true = np.asarray(y_true, dtype=int); y_pred = np.asarray(y_pred, dtype=int); y_score = np.asarray(y_score, dtype=float)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    two_classes = len(np.unique(y_true)) == 2
    return {
        "n_test_windows": int(len(y_true)),
        "positive_prevalence": float(np.mean(y_true == 1)) if len(y_true) else np.nan,
        "DoS_precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "DoS_recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "DoS_F1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "macro_F1": float(f1_score(y_true, y_pred, average="macro", labels=[0, 1], zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)) if two_classes else np.nan,
        "PR_AUC_average_precision": float(average_precision_score(y_true, y_score)) if np.any(y_true == 1) else np.nan,
        "ROC_AUC": float(roc_auc_score(y_true, y_score)) if two_classes else np.nan,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "FPR": float(fp / (fp + tn)) if (fp + tn) > 0 else np.nan,
        "batch_predict_seconds": float(predict_seconds),
        "latency_ms_per_window_batch": float(1000 * predict_seconds / len(y_true)) if len(y_true) else np.nan,
        "throughput_windows_per_s_batch": float(len(y_true) / predict_seconds) if predict_seconds > 0 else np.nan,
        "latency_mode": "batch_predict; not single-window online latency",
    }


def make_grid_search(name, estimator, X_train, y_train, groups):
    if name not in PARAM_GRIDS or len(np.unique(groups)) < 3 or len(np.unique(y_train)) < 2:
        return None
    n_splits = min(3, len(np.unique(groups)))
    cv = GroupKFold(n_splits=n_splits)
    search = GridSearchCV(estimator, PARAM_GRIDS[name], scoring="f1", cv=cv, n_jobs=1, refit=True, error_score="raise", return_train_score=False)
    search.fit(X_train, y_train, groups=groups)
    return search


def run_outer_evaluation(df, feature_columns, split_col, model_names, experiment_name, tuning_mode="fixed"):
    metric_rows, prediction_frames, split_manifest, importance_rows = [], [], [], []
    split_values = sorted(df[split_col].dropna().astype(str).unique().tolist())
    if len(split_values) < 2:
        raise ValueError(f"Không đủ nhóm cho {split_col} split: {split_values}")
    for fold_no, test_value in enumerate(split_values, start=1):
        test_mask = df[split_col].astype(str).eq(test_value)
        train_df, test_df = df.loc[~test_mask].copy(), df.loc[test_mask].copy()
        if train_df.empty or test_df.empty:
            continue
        y_train = train_df["y"].astype(int).to_numpy()
        y_test = test_df["y"].astype(int).to_numpy()
        if len(np.unique(y_train)) < 2:
            print(f"SKIP fold {test_value}: training partition chỉ có một lớp")
            continue
        X_train = train_df[feature_columns].replace([np.inf, -np.inf], np.nan)
        X_test = test_df[feature_columns].replace([np.inf, -np.inf], np.nan)
        # Train-only schema gate: a field with no observed value in this fold's train data
        # cannot be imputed or selected using held-out test information.
        fit_feature_columns = [c for c in X_train.columns if X_train[c].notna().any()]
        if not fit_feature_columns:
            print(f"SKIP fold {test_value}: không có feature hữu dụng trong training partition")
            continue
        X_train = X_train[fit_feature_columns]
        X_test = X_test[fit_feature_columns]
        groups_train = train_df["condition_group"].astype(str).to_numpy()
        split_manifest.append({
            "experiment": experiment_name, "split_col": split_col, "held_out_value": str(test_value),
            "train_runs": sorted(train_df["run_id"].astype(str).unique().tolist()),
            "test_runs": sorted(test_df["run_id"].astype(str).unique().tolist()),
            "n_train_windows": int(len(train_df)), "n_test_windows": int(len(test_df)),
            "train_classes": pd.Series(y_train).value_counts().to_dict(),
            "test_classes": pd.Series(y_test).value_counts().to_dict(),
            "inner_group_count": int(len(np.unique(groups_train))),
        })
        print(f"\n{experiment_name} | fold {fold_no}/{len(split_values)} | {split_col}={test_value} | train={len(train_df):,} test={len(test_df):,}")
        for name in model_names:
            try:
                estimator = build_model(name, for_search=(tuning_mode == "grouped_cv"), n_train=len(X_train))
                best_params = {}
                if tuning_mode == "grouped_cv" and name in PARAM_GRIDS:
                    search = make_grid_search(name, estimator, X_train, y_train, groups_train)
                    if search is not None:
                        estimator = search.best_estimator_
                        best_params = search.best_params_
                    else:
                        estimator.fit(X_train, y_train)
                else:
                    estimator.fit(X_train, y_train)

                t0 = time.perf_counter()
                y_pred = estimator.predict(X_test)
                predict_seconds = time.perf_counter() - t0
                y_score = continuous_scores(estimator, X_test)
                metrics = compute_metrics(y_test, y_pred, y_score, predict_seconds)
                metric_rows.append({
                    "experiment": experiment_name, "split_col": split_col, "fold": fold_no,
                    "held_out_value": str(test_value), "model": name, "tuning_mode": tuning_mode,
                    "best_params": json.dumps(best_params, sort_keys=True), **metrics,
                })
                pred = test_df[["run_id", "mission_id", "channel_id", "attack_id", "seed", "condition_group", "window_start_epoch_s", "window_size_s", "y"]].copy()
                pred = pred.rename(columns={"y": "y_true"})
                pred["experiment"] = experiment_name
                pred["split_col"] = split_col
                pred["held_out_value"] = str(test_value)
                pred["model"] = name
                pred["y_pred"] = np.asarray(y_pred, dtype=int)
                pred["y_score"] = np.asarray(y_score, dtype=float)
                prediction_frames.append(pred)

                model_artifact = ""
                if SAVE_MODEL_ARTIFACTS:
                    artifact_rel = Path("models") / f"{norm_key(experiment_name)}__{norm_key(split_col)}_{norm_key(test_value)}__{norm_key(name)}.joblib"
                    joblib.dump(estimator, OUTPUT_DIR / artifact_rel, compress=3)
                    model_artifact = str(artifact_rel)
                    metric_rows[-1]["model_artifact"] = model_artifact
                fitted_clf = estimator.named_steps["clf"]
                if hasattr(fitted_clf, "feature_importances_"):
                    for feat, imp in zip(fit_feature_columns, fitted_clf.feature_importances_):
                        importance_rows.append({"experiment": experiment_name, "fold": fold_no, "held_out_value": str(test_value), "model": name, "feature": feat, "importance": float(imp)})
            except Exception as e:
                print(f"ERROR model={name}, fold={test_value}: {type(e).__name__}: {e}")
                metric_rows.append({"experiment": experiment_name, "split_col": split_col, "fold": fold_no,
                                    "held_out_value": str(test_value), "model": name, "tuning_mode": tuning_mode,
                                    "error": f"{type(e).__name__}: {e}"})
    metrics_df = pd.DataFrame(metric_rows)
    preds_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
    manifest_df = pd.DataFrame(split_manifest)
    importance_df = pd.DataFrame(importance_rows)
    return metrics_df, preds_df, manifest_df, importance_df

print("Baselines and evaluation helpers loaded.")

## 7. Thí nghiệm chính — 1 giây, network-only, Leave-One-Mission-Out

Mỗi outer fold giữ lại toàn bộ một mission. Trong mỗi fold, tất cả cửa sổ từ các PCAP của mission đó chỉ nằm ở test. Nếu bật `grouped_cv`, inner tuning chia theo `condition_group = mission × channel × seed`, nên run benign và DoS tương ứng không bị tách giữa inner train/validation. Không có row-level random split.

In [ ]:
primary_df = features_all[features_all["window_size_s"] == PRIMARY_WINDOW].copy().reset_index(drop=True)
# Numeric network features only; provenance, label-overlap and grouping columns are explicitly excluded.
feature_cols = [c for c in primary_df.columns if c not in PROVENANCE_COLS and pd.api.types.is_numeric_dtype(primary_df[c])]
# Keep the predeclared schema here. If any column is all-missing in a particular train fold,
# it is removed inside run_outer_evaluation using TRAIN data only.

if primary_df["mission_id"].nunique() < 3:
    print("WARNING: có ít hơn 3 mission; Leave-One-Mission-Out sẽ chỉ chạy trên các mission hiện có.")
if set(primary_df["attack_id"].dropna().unique()) != EXPECTED_ATTACKS:
    print("WARNING: lớp thực tế không đầy đủ; không được tuyên bố kết quả nhị phân tổng quát nếu thiếu Benign/DoS.")

main_metrics, main_oof, main_splits, main_importance = run_outer_evaluation(
    primary_df, feature_cols, split_col="mission_id", model_names=MODEL_NAMES,
    experiment_name="primary_1s_LOMO_network_only", tuning_mode=TUNING_MODE
)
main_metrics.to_csv(OUTPUT_DIR / "model_metrics_per_fold.csv", index=False)
main_metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)  # alias matching the test plan deliverable name
main_oof.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)
main_splits.to_json(OUTPUT_DIR / "split_manifest.json", orient="records", indent=2)
if not main_importance.empty:
    main_importance.to_csv(OUTPUT_DIR / "tree_feature_importance_by_fold.csv", index=False)

print("\nPer-fold metrics:")
display(main_metrics)

## 8. Tổng hợp metric, confusion matrix, PR curve và feature importance

Báo cáo mean ± standard deviation qua outer folds. Với LOMO, chỉ có ba mission nên độ lệch chuẩn giữa fold cần được diễn giải thận trọng, không phải CI chính xác cho mọi UAV/mission ngoài thực tế. PR-AUC được tính bằng average precision; trong bài viết hãy nêu rõ positive prevalence đi kèm.

In [ ]:
METRICS_TO_SUMMARIZE = [
    "DoS_precision", "DoS_recall", "DoS_F1", "macro_F1", "balanced_accuracy",
    "PR_AUC_average_precision", "ROC_AUC", "FPR", "latency_ms_per_window_batch", "throughput_windows_per_s_batch"
]
valid_metrics = main_metrics[main_metrics.get("error", pd.Series(index=main_metrics.index, dtype=object)).isna()].copy() if not main_metrics.empty else main_metrics.copy()
agg_spec = {m: ["mean", "std"] for m in METRICS_TO_SUMMARIZE if m in valid_metrics.columns}
if not valid_metrics.empty and agg_spec:
    summary = valid_metrics.groupby("model").agg(agg_spec)
    summary.columns = [f"{a}_{b}" for a, b in summary.columns]
    summary = summary.reset_index().sort_values("DoS_F1_mean", ascending=False)
    summary.to_csv(OUTPUT_DIR / "model_metrics_summary.csv", index=False)
    display(summary)
else:
    summary = pd.DataFrame()
    print("Chưa có metric hợp lệ để tổng hợp; xem log lỗi từng fold.")

# Pooled outer-fold confusion matrices and PR curves; these use only out-of-fold test predictions.
if not main_oof.empty:
    for model_name in main_oof["model"].unique():
        p = main_oof[main_oof["model"] == model_name]
        cm = confusion_matrix(p["y_true"], p["y_pred"], labels=[0, 1])
        fig, ax = plt.subplots(figsize=(4.6, 4.0))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Benign (0)", "DoS (1)"])
        disp.plot(ax=ax, colorbar=False, values_format="d")
        ax.set_title(f"Pooled OOF confusion matrix — {model_name}")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"confusion_matrix_{norm_key(model_name)}.png", dpi=160)
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5.5))
    plotted = 0
    for model_name in main_oof["model"].unique():
        p = main_oof[main_oof["model"] == model_name]
        if p["y_true"].nunique() < 2 or not p["y_score"].notna().all():
            continue
        PrecisionRecallDisplay.from_predictions(p["y_true"], p["y_score"], name=model_name, ax=ax)
        plotted += 1
    if plotted:
        ax.set_title("Pooled out-of-fold precision–recall curves (primary 1s LOMO)")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "pr_curve_pooled_oof.png", dpi=180)
    plt.close(fig)

if not main_importance.empty:
    imp_summary = main_importance.groupby(["model", "feature"], as_index=False)["importance"].mean()
    imp_summary.to_csv(OUTPUT_DIR / "tree_feature_importance_mean.csv", index=False)
    for model_name in ["RandomForest", "ExtraTrees", "HistGradientBoosting"]:
        d = imp_summary[imp_summary["model"] == model_name].nlargest(20, "importance")
        if d.empty: continue
        fig, ax = plt.subplots(figsize=(8, 6))
        d = d.sort_values("importance")
        ax.barh(d["feature"], d["importance"])
        ax.set_title(f"Top feature importances — {model_name} (mean across outer-fold fits)")
        ax.set_xlabel("Training-fit feature importance")
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / f"feature_importance_{norm_key(model_name)}.png", dpi=160)
        plt.close(fig)

print("Figures saved under:", FIGURE_DIR)

## 9. Thí nghiệm phụ: cửa sổ 2/5 giây và Leave-One-Channel-Scenario-Out

Các thí nghiệm phụ dùng lại định nghĩa feature/label và không thay thế kết quả chính. Để tránh tăng runtime quá mức, notebook mặc định chạy các mô hình có DoS-F1 mean cao nhất trong primary result (tối đa `TOP_K_SECONDARY`). Nếu kết quả chính lỗi hoặc thiếu metric, cell sẽ báo để kiểm tra thay vì tự chọn mô hình.

In [ ]:
secondary_outputs = []
if RUN_SECONDARY_EVALUATION:
    if summary.empty or "DoS_F1_mean" not in summary.columns:
        print("BỎ QUA secondary: chưa có summary primary hợp lệ.")
    else:
        top_models = summary.dropna(subset=["DoS_F1_mean"]).head(TOP_K_SECONDARY)["model"].tolist()
        print("Top models selected using primary outer-fold mean DoS_F1:", top_models)
        for W in [w for w in WINDOW_SIZES if w != PRIMARY_WINDOW]:
            d = features_all[features_all["window_size_s"] == W].copy().reset_index(drop=True)
            if d.empty:
                print(f"BỎ QUA {W}s: không có windows.")
                continue
            fcols = [c for c in d.columns if c not in PROVENANCE_COLS and pd.api.types.is_numeric_dtype(d[c])]
            m, oof, splits, imps = run_outer_evaluation(
                d, fcols, split_col="mission_id", model_names=top_models,
                experiment_name=f"sensitivity_{W}s_LOMO_network_only", tuning_mode=TUNING_MODE
            )
            m.to_csv(OUTPUT_DIR / f"model_metrics_{W}s_LOMO.csv", index=False)
            oof.to_csv(OUTPUT_DIR / f"oof_predictions_{W}s_LOMO.csv", index=False)
            splits.to_json(OUTPUT_DIR / f"split_manifest_{W}s_LOMO.json", orient="records", indent=2)
            secondary_outputs.append(m)

        # Secondary channel holdout is run at 1s on the same selected models.
        d = features_all[features_all["window_size_s"] == PRIMARY_WINDOW].copy().reset_index(drop=True)
        fcols = [c for c in d.columns if c not in PROVENANCE_COLS and pd.api.types.is_numeric_dtype(d[c])]
        if d["channel_id"].nunique() >= 2:
            m, oof, splits, imps = run_outer_evaluation(
                d, fcols, split_col="channel_id", model_names=top_models,
                experiment_name="secondary_1s_leave_one_channel_out", tuning_mode=TUNING_MODE
            )
            m.to_csv(OUTPUT_DIR / "model_metrics_channel_holdout.csv", index=False)
            oof.to_csv(OUTPUT_DIR / "oof_predictions_channel_holdout.csv", index=False)
            splits.to_json(OUTPUT_DIR / "split_manifest_channel_holdout.json", orient="records", indent=2)
            secondary_outputs.append(m)
        else:
            print("BỎ QUA channel holdout: không đủ channel_id.")
        if secondary_outputs:
            sec = pd.concat(secondary_outputs, ignore_index=True)
            sec.to_csv(OUTPUT_DIR / "secondary_metrics_all.csv", index=False)
            display(sec)
else:
    print("RUN_SECONDARY_EVALUATION=False; chỉ chạy protocol chính.")

## 10. Tuỳ chọn: network + telemetry fusion (chỉ bật sau khi xác minh clock)

Đừng bật ngay nếu telemetry dùng simulation time trong khi PCAP dùng epoch. Cell sau đọc cột timestamp dạng epoch hoặc datetime và chỉ aggregate mẫu telemetry nằm **bên trong đúng cửa sổ** (không lấy sample tương lai gần nhất). Chỉ dùng speed, altitude tương đối, khoảng cách GCS và mission progress/is-in-air nếu tên cột có thể nhận diện rõ; không tự dùng latitude/longitude hoặc timestamp làm feature.

Trước khi bật, kiểm tra `flight_telemetry.csv` của vài run đối chiếu với `pcap_first_epoch_s`/`pcap_last_epoch_s` và xác minh offset đồng hồ. Nếu không có cột epoch/datetime đồng bộ hoặc không có feature phù hợp, giữ `RUN_TELEMETRY_FUSION=False` và ghi rõ lý do.

In [ ]:
TELEMETRY_COL_ALIASES = {
    "speed": ["speed", "ground_speed", "groundspeed", "velocity_m_s", "speed_m_s"],
    "altitude_relative_m": ["altitude_relative_m", "relative_altitude", "rel_altitude", "altitude_rel_m"],
    "distance_to_gcs_m": ["distance_to_gcs_m", "distance_gcs_m", "gcs_distance_m", "distance_to_gcs"],
    "mission_progress": ["mission_progress", "progress", "mission_progress_ratio"],
    "is_in_air": ["is_in_air", "in_air", "armed_in_air"],
}
TIMESTAMP_ALIASES = ["timestamp_epoch_s", "epoch_s", "timestamp", "time_utc", "utc_timestamp", "unix_time", "timestamp_s"]


def normalize_telemetry_timestamp(series):
    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().mean() > 0.8 and numeric.notna().any():
        med = float(numeric.dropna().median())
        if med < 100_000_000:  # likely simulator-relative seconds; do not auto-align to epoch
            raise ValueError(f"Telemetry timestamp numeric median={med:g} không giống epoch seconds.")
        if med > 1e17:  # pandas datetime64/int64 nanoseconds
            return numeric.astype(float) / 1e9
        if med > 1e11:  # Unix milliseconds
            return numeric.astype(float) / 1e3
        return numeric.astype(float)
    dt = pd.to_datetime(series, errors="coerce", utc=True)
    if dt.notna().mean() < 0.8:
        raise ValueError("Không parse được telemetry timestamp thành epoch/datetime.")
    return dt.astype("int64") / 1e9


def telemetry_features_for_run(run_id, telemetry_path, window_rows):
    if not telemetry_path or not Path(telemetry_path).is_file():
        raise ValueError("missing_telemetry")
    td = pd.read_csv(telemetry_path)
    if td.empty:
        raise ValueError("empty_telemetry")
    colmap = {norm_key(c): c for c in td.columns}
    tscol = next((colmap[norm_key(c)] for c in TIMESTAMP_ALIASES if norm_key(c) in colmap), None)
    if tscol is None:
        raise ValueError(f"No recognized timestamp column. columns={td.columns.tolist()}")
    td["_epoch_s"] = normalize_telemetry_timestamp(td[tscol])
    target_min = float(window_rows["window_start_epoch_s"].min())
    target_max = float((window_rows["window_start_epoch_s"] + window_rows["window_size_s"]).max())
    if float(td["_epoch_s"].max()) < target_min or float(td["_epoch_s"].min()) > target_max:
        raise ValueError("Telemetry timestamp không overlap cửa sổ PCAP; có thể khác clock basis/offset.")
    matched = {}
    for target, aliases in TELEMETRY_COL_ALIASES.items():
        col = next((colmap[norm_key(a)] for a in aliases if norm_key(a) in colmap), None)
        if col is not None:
            td[target] = pd.to_numeric(td[col], errors="coerce")
            matched[target] = target
    if not matched:
        raise ValueError(f"No recognized safe telemetry features. columns={td.columns.tolist()}")
    out = []
    for _, wr in window_rows.iterrows():
        s = float(wr["window_start_epoch_s"]); e = s + float(wr["window_size_s"])
        chunk = td[(td["_epoch_s"] >= s) & (td["_epoch_s"] < e)]
        row = {"run_id": run_id, "window_start_epoch_s": s, "window_size_s": int(wr["window_size_s"])}
        for c in matched:
            vals = chunk[c].dropna().to_numpy(dtype=float)
            for stat in ["mean", "std", "min", "max"]:
                row[f"telemetry_{c}_{stat}"] = float(getattr(np, stat)(vals)) if len(vals) else np.nan
        row["telemetry_sample_count"] = int(len(chunk))
        out.append(row)
    return out, sorted(matched)


if RUN_TELEMETRY_FUSION:
    inv_by_run = inventory.set_index("run_id")
    tele_rows, tele_errors = [], []
    fusion_base = features_all[features_all["window_size_s"] == PRIMARY_WINDOW].copy()
    for run_id, run_windows in fusion_base.groupby("run_id", sort=False):
        tel_path = inv_by_run.loc[run_id, "telemetry_path"]
        try:
            tr, matched_cols = telemetry_features_for_run(run_id, tel_path, run_windows)
            tele_rows.extend(tr)
        except Exception as e:
            tele_errors.append({"run_id": run_id, "reason": f"{type(e).__name__}:{e}"})
    pd.DataFrame(tele_errors).to_csv(OUTPUT_DIR / "telemetry_fusion_exclusions.csv", index=False)
    tele_df = pd.DataFrame(tele_rows)
    if tele_df.empty or tele_errors or len(tele_df) != len(fusion_base):
        print("Fusion không chạy để tránh so sánh trên split khác: một hoặc nhiều run chưa có telemetry timestamp/feature đồng bộ.")
        print("Xem telemetry_fusion_exclusions.csv và kiểm tra clock basis/offset. Không tự drop các run lỗi rồi so với network-only.")
    else:
        fusion_df = fusion_base.merge(tele_df, on=["run_id", "window_start_epoch_s", "window_size_s"], how="left", validate="one_to_one")
        if len(fusion_df) != len(fusion_base):
            raise RuntimeError("Fusion row count khác network-only; dừng để tránh split không tương đương.")
        fusion_feature_cols = [c for c in feature_cols if c in fusion_df.columns] + [c for c in fusion_df.columns if c.startswith("telemetry_") and c != "telemetry_sample_count"]
        print("Fusion feature columns:", len(fusion_feature_cols), "| windows retained:", len(fusion_df), "| runs:", fusion_df.run_id.nunique())
        m, oof, splits, imps = run_outer_evaluation(
            fusion_df, fusion_feature_cols, split_col="mission_id",
            model_names=[m for m in ["LogisticRegression", "RandomForest", "ExtraTrees"] if m in MODEL_NAMES],
            experiment_name="optional_1s_LOMO_network_plus_telemetry", tuning_mode=TUNING_MODE
        )
        m.to_csv(OUTPUT_DIR / "model_metrics_network_plus_telemetry.csv", index=False)
        oof.to_csv(OUTPUT_DIR / "oof_predictions_network_plus_telemetry.csv", index=False)
        splits.to_json(OUTPUT_DIR / "split_manifest_network_plus_telemetry.json", orient="records", indent=2)
else:
    print("Telemetry fusion tắt theo cấu hình. Chưa có kết luận nào về lợi ích sensor fusion.")

## 11. Tuỳ chọn: Isolation Forest benign-only

Đây là novelty/anomaly experiment riêng, không nằm trong bảng supervised baseline. Cấu hình dùng Isolation Forest fit chỉ trên cửa sổ train `y=0`; `contamination=0.05` là ngưỡng tỷ lệ anomaly định trước, không điều chỉnh theo outer-test. Chỉ chạy khi mọi fold có đủ benign train và dữ liệu thực sự cần câu hỏi anomaly detection. Với chiến dịch hiện tại chỉ có một họ attack DoS, không được diễn giải thành khả năng phát hiện zero-day tổng quát.

In [ ]:
if RUN_ISOLATION_FOREST:
    iso_rows, iso_preds = [], []
    d = features_all[features_all["window_size_s"] == PRIMARY_WINDOW].copy().reset_index(drop=True)
    fcols = [c for c in d.columns if c not in PROVENANCE_COLS and pd.api.types.is_numeric_dtype(d[c])]
    for fold_no, test_mission in enumerate(sorted(d["mission_id"].dropna().astype(str).unique()), start=1):
        test_mask = d["mission_id"].astype(str).eq(test_mission)
        tr, te = d.loc[~test_mask].copy(), d.loc[test_mask].copy()
        benign_train = tr[tr["y"] == 0]
        if benign_train.empty or te.empty:
            continue
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED, n_jobs=-1)),
        ])
        Xb = benign_train[fcols].replace([np.inf, -np.inf], np.nan)
        Xt = te[fcols].replace([np.inf, -np.inf], np.nan)
        # Remove all-missing fields based on benign training windows only.
        fit_cols = [c for c in Xb.columns if Xb[c].notna().any()]
        if not fit_cols: continue
        Xb, Xt = Xb[fit_cols], Xt[fit_cols]
        pipe.fit(Xb)
        t0 = time.perf_counter(); pred0 = pipe.predict(Xt); secs = time.perf_counter() - t0
        yhat = (pred0 == -1).astype(int)
        # Larger anomaly score means more anomalous.
        score = -pipe.decision_function(Xt)
        met = compute_metrics(te["y"].astype(int), yhat, score, secs)
        iso_rows.append({"experiment":"exploratory_IsolationForest_benign_only", "held_out_mission":test_mission,
                         "fit_benign_windows":len(benign_train), "model":"IsolationForest_contamination_0.05", **met})
        q = te[["run_id", "mission_id", "channel_id", "attack_id", "seed", "window_start_epoch_s", "y"]].copy().rename(columns={"y":"y_true"})
        q["held_out_mission"] = test_mission; q["y_pred"] = yhat; q["anomaly_score"] = score
        iso_preds.append(q)
    pd.DataFrame(iso_rows).to_csv(OUTPUT_DIR / "isolation_forest_metrics.csv", index=False)
    if iso_preds: pd.concat(iso_preds, ignore_index=True).to_csv(OUTPUT_DIR / "isolation_forest_oof_predictions.csv", index=False)
else:
    print("Isolation Forest tắt theo cấu hình.")

## 12. Khóa cấu hình, log phiên chạy và đóng gói kết quả

Trước khi nộp hội nghị, lưu cùng bản code đã chạy: cấu hình, phiên bản thư viện, inventory, hashes, exclusions, fold manifest, out-of-fold predictions và metrics. Mọi thay đổi về feature, window rule, mô hình hoặc split sau khi xem outer-test phải được ghi ngày/lý do trong change log.

In [ ]:
config = {
    "protocol_version": "1.0",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "campaign_dir": str(CAMPAIGN_DIR),
    "expected_runs_nominal": EXPECTED_RUNS,
    "observed_metadata_records": int(len(inventory)),
    "window_sizes_s": WINDOW_SIZES,
    "primary_window_s": PRIMARY_WINDOW,
    "label_rule": "y=1 when merged attack intervals overlap >= 50% of full epoch-aligned window [t,t+W); benign runs always y=0",
    "edge_window_rule": "keep only full W-second epoch-aligned bins within first/last packet timestamp span",
    "features": feature_cols,
    "excluded_model_columns": sorted(PROVENANCE_COLS),
    "gcs_ip_for_directional_features": GCS_IP,
    "sha256_raw_inputs_enabled": HASH_RAW_INPUTS,
    "save_model_artifacts": SAVE_MODEL_ARTIFACTS,
    "allow_unknown_status_after_manual_check": ALLOW_UNKNOWN_STATUS_AFTER_MANUAL_CHECK,
    "tuning_mode": TUNING_MODE,
    "primary_split": "Leave-One-Mission-Out; train on other missions; one complete mission held out per fold",
    "inner_group": "mission_id|channel_id|seed",
    "secondary_evaluation_enabled": RUN_SECONDARY_EVALUATION,
    "telemetry_fusion_enabled": RUN_TELEMETRY_FUSION,
    "isolation_forest_enabled": RUN_ISOLATION_FOREST,
    "models": MODEL_NAMES,
    "seed": SEED,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
}
(OUTPUT_DIR / "experiment_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

log_lines = [
    f"Run completed UTC: {datetime.now(timezone.utc).isoformat()}",
    f"Campaign dir: {CAMPAIGN_DIR}",
    f"Inventory records: {len(inventory)}",
    f"Unique runs in primary windows: {primary_df['run_id'].nunique() if 'primary_df' in globals() and not primary_df.empty else 0}",
    f"Excluded runs: {len(excluded_df)}",
    f"Primary window: {PRIMARY_WINDOW}s",
    f"Tuning mode: {TUNING_MODE}",
    f"Primary metrics rows: {len(main_metrics) if 'main_metrics' in globals() else 0}",
    f"Primary OOF prediction rows: {len(main_oof) if 'main_oof' in globals() else 0}",
    "NOTE: measured model outputs must be interpreted under simulated/emulated PX4 SITL/Gazebo + MAVLink/Containernet scope.",
    "NOTE: this campaign does not establish calibrated physical RF propagation, an RF-to-C2 mapping, or a real-aircraft IDS result.",
]
(OUTPUT_DIR / "run_log.txt").write_text("\n".join(log_lines), encoding="utf-8")

# Hash generated feature files after writing them.
feature_hash_rows = []
for p in sorted(OUTPUT_DIR.glob("features_*s.csv")):
    feature_hash_rows.append({"file": p.name, "bytes": p.stat().st_size, "sha256": sha256_file(p)})
if feature_hash_rows:
    pd.DataFrame(feature_hash_rows).to_csv(OUTPUT_DIR / "generated_feature_hashes.csv", index=False)

zip_path = Path("/kaggle/working/uav_c2_baseline_results.zip")
import zipfile
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(Path("results") / p.relative_to(OUTPUT_DIR)))
print("\nRESULTS PACKAGED:", zip_path)
print("ZIP bytes:", zip_path.stat().st_size)
print("Files included:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file(): print(f" - {p.relative_to(OUTPUT_DIR)} ({p.stat().st_size:,} bytes)")
try:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))
except Exception:
    pass

## 13. Checklist khi đọc kết quả cho bài hội nghị

1. Đọc `excluded_runs.csv`, `pcap_quality_by_run.csv`, `data_quality_report.txt` trước khi nhìn F1. Giải thích mọi run bị loại; không chỉ báo số run dự kiến.
2. Dùng `model_metrics_per_fold.csv` và `model_metrics_summary.csv` cho kết quả supervised chính. Báo cáo từng held-out mission và mean ± SD qua 3 outer folds.
3. Dùng `oof_predictions.csv` tạo pooled confusion matrix/PR curve; nhấn mạnh đây là pooled out-of-fold predictions, không phải random split.
4. Báo cáo `positive_prevalence`, DoS recall/F1, macro-F1, PR-AUC, FPR; accuracy đơn độc là không đủ. Latency hiện là **batch prediction latency**, không phải single-window online latency.
5. So sánh 2/5 giây và channel holdout như thí nghiệm phụ. Fusion chỉ được báo khi clock alignment được chứng minh. Isolation Forest chỉ là novelty/anomaly trên họ DoS đã quan sát.
6. Mô tả dữ liệu là mô phỏng/emulation. Không tuyên bố radio propagation đã được hiệu chuẩn, không gọi kết quả là triển khai trên UAV thật, và không tuyên bố multi-class attack identification khi chỉ có benign/DoS.

### Lấy kết quả từ Kaggle

Sau khi chạy xong, tải file `uav_c2_baseline_results.zip` trong phần Output của notebook (hoặc bấm liên kết cuối cell cuối). Các file quan trọng nhất: `dataset_inventory.csv`, `data_quality_report.txt`, `excluded_runs.csv`, `features_1s.csv`, `model_metrics_per_fold.csv`, `model_metrics_summary.csv`, `oof_predictions.csv`, `split_manifest.json`, `experiment_config.json`, `input_hashes.csv` và thư mục `figures/`.